# DAC metric depth baseline on the project cameras

This notebook runs the original **Depth Any Camera (DAC)** indoor
ResNet101 model on the calibrated G1_A fisheye images. It mirrors the
UniDAC notebook's frame selection, output filenames, visualization
range, timing metadata, person tracking, and physical-sensor evaluation
so the two runs can be compared directly.

Add a GitHub token that can read the private project repository to
Colab **Secrets** as `GITHUB_TOKEN`, enable notebook access, and never
paste the token into a code cell.

Before running, choose **Runtime → Change runtime type → GPU**. Run the
cells from top to bottom. The official DAC source is pinned to commit
`371ee299` and the checkpoint is cached in Google Drive.

## 1. Install pinned runtime dependencies

Colab supplies PyTorch and CUDA. The first run installs a NumPy/OpenCV
combination that is binary-compatible with DAC and restarts the kernel
once. After Colab reconnects, choose **Runtime → Run all** again.

In [ ]:
import subprocess
import sys
from pathlib import Path

setup_marker = Path("/content/.dac_numpy_1_26_4_yolo26_v1_ready")
packages = [
    "numpy==1.26.4",
    "opencv-python==4.11.0.86",
    "einops>=0.6",
    "timm>=0.9",
    "huggingface_hub>=0.20",
    "matplotlib>=3.8",
    "scipy>=1.10",
    "ultralytics==8.4.67",
    "lap==0.5.12",
]

if not setup_marker.exists():
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "--no-cache-dir", "-q", *packages]
    )
    setup_marker.write_text("numpy==1.26.4\n")
    print(
        "Dependencies installed. Colab is restarting the Python kernel once. "
        "After it reconnects, choose Runtime → Run all."
    )
    import IPython

    IPython.Application.instance().kernel.do_shutdown(True)
else:
    print("Pinned DAC and tracking dependencies are ready.")

## 2. Experiment settings

The defaults target the completed UniDAC protocol: all 133 consecutive
`recording1/G1_A` frames. Keep `DAC_VARIANT` and `DAC_FORWARD_SIZE`
unchanged for the official indoor fisheye baseline.

In [ ]:
from pathlib import Path

PROJECT_REPO_URL = "https://github.com/esthy13/monocular-depth-estimation.git"
PROJECT_REF = "main"
PROJECT_REPO_IS_PRIVATE = True
GITHUB_TOKEN_SECRET = "GITHUB_TOKEN"
DAC_COMMIT = "371ee299429257bb9a27d1e23b7dc53670e37023"
DAC_VARIANT = "dac-indoor-resnet101"
DAC_FORWARD_SIZE = (500, 750)

PROJECT_DIR = Path("/content/monocular-depth-estimation")
DAC_DIR = PROJECT_DIR / "third_party" / "depth_any_camera"

DATA_DIR = Path("/content/drive/MyDrive/cv_project_data")
OUTPUT_DIR = Path("/content/drive/MyDrive/cv_project_outputs/dac")
CHECKPOINT_CACHE_DIR = Path("/content/drive/MyDrive/cv_project_cache/dac")
DETECTOR_WEIGHTS = Path("/content/drive/MyDrive/cv_project_cache/yolo/yolo26n-seg.pt")

RECORDING = "recording1"
SENSOR = "G1_A"
IMAGE_INDEX = 0
VISUALIZATION_DEPTH_RANGE_M = (0.5, 10.0)

# One timed run per batch frame records pipeline latency. The controlled
# benchmark later must use WARMUP_RUNS=2 and TIMED_RUNS=10.
WARMUP_RUNS = 0
TIMED_RUNS = 1


## 3. Mount Drive and verify the GPU/data

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA is not available. In Colab choose Runtime → Change runtime type → GPU, "
        "then reconnect and run all cells again."
    )
if not (DATA_DIR / "intrinsic.json").is_file():
    raise FileNotFoundError(
        f"Could not find {DATA_DIR / 'intrinsic.json'}. Upload cv_project_data "
        "to Drive or edit DATA_DIR in the settings cell."
    )

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_CACHE_DIR.mkdir(parents=True, exist_ok=True)
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"PyTorch: {torch.__version__}; CUDA runtime: {torch.version.cuda}")
print(f"Data: {DATA_DIR}")
print(f"Outputs: {OUTPUT_DIR}")

## 4. Fetch the project and pinned DAC source

The project credential is passed only to the Git process and is not
saved in the notebook, clone URL, or Git configuration. The public DAC
source is downloaded from the exact pinned commit.

In [ ]:
import base64
import os
import shutil
import subprocess
import tarfile
import urllib.request

from google.colab import userdata

def project_git_environment():
    environment = os.environ.copy()
    if not PROJECT_REPO_IS_PRIVATE:
        return environment
    try:
        github_token = userdata.get(GITHUB_TOKEN_SECRET)
    except Exception as error:
        raise RuntimeError(
            f"Add a GitHub token named {GITHUB_TOKEN_SECRET!r} in Colab Secrets "
            "(key icon), enable notebook access, and rerun this cell."
        ) from error
    if not github_token:
        raise RuntimeError(f"Colab Secret {GITHUB_TOKEN_SECRET!r} is empty.")
    basic_credential = base64.b64encode(
        f"x-access-token:{github_token}".encode()
    ).decode()
    environment.update({
        "GIT_CONFIG_COUNT": "1",
        "GIT_CONFIG_KEY_0": "http.https://github.com/.extraheader",
        "GIT_CONFIG_VALUE_0": f"AUTHORIZATION: basic {basic_credential}",
    })
    return environment

git_environment = project_git_environment()

def run_command(arguments):
    subprocess.check_call([str(item) for item in arguments], env=git_environment)

if not (PROJECT_DIR / ".git").is_dir():
    run_command([
        "git", "clone", "--branch", PROJECT_REF, "--single-branch",
        PROJECT_REPO_URL, PROJECT_DIR,
    ])
else:
    run_command(["git", "-C", PROJECT_DIR, "fetch", "origin", PROJECT_REF])
    run_command(["git", "-C", PROJECT_DIR, "checkout", PROJECT_REF])
    run_command([
        "git", "-C", PROJECT_DIR, "merge", "--ff-only", f"origin/{PROJECT_REF}"
    ])

project_commit = subprocess.check_output(
    ["git", "-C", PROJECT_DIR, "rev-parse", "HEAD"], text=True
).strip()
git_environment.pop("GIT_CONFIG_VALUE_0", None)

dac_marker = DAC_DIR / ".pinned_commit"
installed_dac_commit = (
    dac_marker.read_text().strip() if dac_marker.is_file() else None
)
if DAC_DIR.exists() and installed_dac_commit != DAC_COMMIT:
    raise RuntimeError(
        f"{DAC_DIR} exists but is not the requested pinned source. Start a fresh "
        "Colab runtime or remove only that directory and rerun this cell."
    )
if not DAC_DIR.exists():
    archive_path = Path("/content/dac-source.tar.gz")
    staging_dir = Path("/content/dac-source")
    if staging_dir.exists():
        shutil.rmtree(staging_dir)
    staging_dir.mkdir(parents=True)
    archive_url = (
        f"https://github.com/yuliangguo/depth_any_camera/archive/{DAC_COMMIT}.tar.gz"
    )
    print("Downloading pinned DAC source ...")
    urllib.request.urlretrieve(archive_url, archive_path)
    with tarfile.open(archive_path, "r:gz") as archive:
        archive.extractall(staging_dir)
    extracted_dir = next(path for path in staging_dir.iterdir() if path.is_dir())
    DAC_DIR.parent.mkdir(parents=True, exist_ok=True)
    shutil.move(str(extracted_dir), str(DAC_DIR))
    dac_marker.write_text(DAC_COMMIT + "\n")
    archive_path.unlink(missing_ok=True)
    shutil.rmtree(staging_dir)

print(f"Project commit: {project_commit}")
print(f"DAC commit:     {DAC_COMMIT}")

## 5. Cache the official DAC config and checkpoint

Drive retains the files between sessions; a local VM copy is used for
faster model loading.

In [ ]:
import sys

for import_path in (PROJECT_DIR, DAC_DIR):
    if str(import_path) not in sys.path:
        sys.path.insert(0, str(import_path))

from huggingface_hub import hf_hub_download
from src.depth_models import DepthAnyCamera

config_file, checkpoint_file = DepthAnyCamera.VARIANTS[DAC_VARIANT]
drive_config = Path(hf_hub_download(
    repo_id=DepthAnyCamera.HF_REPO,
    filename=config_file,
    local_dir=str(CHECKPOINT_CACHE_DIR),
))
drive_checkpoint = Path(hf_hub_download(
    repo_id=DepthAnyCamera.HF_REPO,
    filename=checkpoint_file,
    local_dir=str(CHECKPOINT_CACHE_DIR),
))

local_model_dir = Path("/content/checkpoints/dac")
local_model_dir.mkdir(parents=True, exist_ok=True)
local_config = local_model_dir / config_file
local_checkpoint = local_model_dir / checkpoint_file
for drive_path, local_path in (
    (drive_config, local_config),
    (drive_checkpoint, local_checkpoint),
):
    if not local_path.is_file() or local_path.stat().st_size != drive_path.stat().st_size:
        print(f"Copying {drive_path.name} from Drive to the Colab VM ...")
        shutil.copy2(drive_path, local_path)

print(f"Config: {local_config}")
print(f"Checkpoint: {local_checkpoint}")
print(f"Checkpoint size: {local_checkpoint.stat().st_size / 1024**2:.1f} MiB")

## 6. Load DAC once

The official indoor ResNet101 configuration does not use the optional
compiled deformable-attention decoder, so the project adapter can run it
on the standard Colab GPU runtime.

In [ ]:
model = DepthAnyCamera(
    variant=DAC_VARIANT,
    fwd_sz=DAC_FORWARD_SIZE,
    device="cuda",
    config_path=str(local_config),
    checkpoint_path=str(local_checkpoint),
)
model.load()

## 7. Helpers for calibrated inference and reproducible outputs

In [ ]:
import csv
import json
import math
import time

import cv2
import numpy as np

from src.utils import (
    create_fisheye_valid_mask,
    find_rgb_images,
    intrinsics_to_dac_cam_params,
    load_intrinsics,
    parse_timestamp,
    save_depth_visualization,
    save_mask_visualization,
)

intrinsics = load_intrinsics(DATA_DIR / "intrinsic.json")

def camera_geometry(sensor, image_width):
    camera = intrinsics[sensor]
    camera_parameters = intrinsics_to_dac_cam_params(sensor, intrinsics)
    if camera.get("model") == "fisheye":
        crop_wfov = 180.0
    else:
        focal_x = float(camera["K"][0][0])
        crop_wfov = math.degrees(2.0 * math.atan(image_width / (2.0 * focal_x)))
    return camera_parameters, crop_wfov

def timed_prediction(image, warmup_runs=0, timed_runs=1):
    if timed_runs < 1:
        raise ValueError("timed_runs must be at least 1")
    for _ in range(warmup_runs):
        model.predict(image)
    torch.cuda.synchronize()
    timings_ms = []
    depth = None
    for _ in range(timed_runs):
        start = time.perf_counter()
        depth = model.predict(image)
        torch.cuda.synchronize()
        timings_ms.append((time.perf_counter() - start) * 1000.0)
    return depth, timings_ms

def process_frame(recording, sensor, image_index, warmup_runs=0, timed_runs=1):
    images = find_rgb_images(DATA_DIR, sensor_name=sensor, recording=recording)
    if not 0 <= image_index < len(images):
        raise IndexError(
            f"image_index={image_index} is outside 0..{len(images) - 1} for "
            f"{recording}/{sensor}"
        )
    image_path = images[image_index]
    image = cv2.imread(str(image_path))
    if image is None:
        raise RuntimeError(f"OpenCV could not read {image_path}")

    camera_parameters, crop_wfov = camera_geometry(sensor, image.shape[1])
    model.set_camera(camera_parameters, crop_wfov)
    depth, timings_ms = timed_prediction(image, warmup_runs, timed_runs)

    valid_mask = np.isfinite(depth) & (depth > 0)
    if intrinsics[sensor].get("model") == "fisheye":
        K = intrinsics[sensor]["K"]
        lens_mask = create_fisheye_valid_mask(image, center=(K[0][2], K[1][2]))
        valid_mask &= lens_mask.astype(bool)
    if not np.any(valid_mask):
        raise RuntimeError("DAC returned no valid positive depth pixels.")
    depth = depth.astype(np.float32, copy=True)
    depth[~valid_mask] = np.nan

    frame_dir = OUTPUT_DIR / recording / sensor
    frame_dir.mkdir(parents=True, exist_ok=True)
    stem = f"{recording}_{sensor}_{image_index:06d}"
    rgb_path = frame_dir / f"{stem}_rgb.jpg"
    raw_path = frame_dir / f"{stem}_depth_raw.npy"
    mask_path = frame_dir / f"{stem}_mask.png"
    visualization_path = frame_dir / f"{stem}_depth.png"
    metadata_path = frame_dir / f"{stem}_metadata.json"

    cv2.imwrite(str(rgb_path), image)
    np.save(raw_path, depth)
    save_mask_visualization(valid_mask.astype(np.uint8), mask_path)
    save_depth_visualization(
        depth,
        visualization_path,
        rgb_image=image,
        valid_mask=valid_mask,
        invert=True,
        value_range=VISUALIZATION_DEPTH_RANGE_M,
        depth_unit="m",
        quantity_label="Metric depth",
    )

    valid_depth = depth[valid_mask]
    try:
        timestamp_seconds = parse_timestamp(image_path)
    except ValueError:
        timestamp_seconds = None
    metadata = {
        "model": "DAC",
        "variant": DAC_VARIANT,
        "checkpoint_repo": DepthAnyCamera.HF_REPO,
        "checkpoint_file": checkpoint_file,
        "dac_commit": DAC_COMMIT,
        "project_commit": project_commit,
        "depth_definition": "metric Euclidean ray distance",
        "depth_unit": "metre",
        "visualization_depth_range_m": list(VISUALIZATION_DEPTH_RANGE_M),
        "recording": recording,
        "sensor": sensor,
        "image_index": image_index,
        "image_file": image_path.name,
        "timestamp_seconds": timestamp_seconds,
        "image_width": image.shape[1],
        "image_height": image.shape[0],
        "camera_model": intrinsics[sensor].get("model"),
        "requested_crop_wfov_degrees": crop_wfov,
        "projection": model.last_projection_metadata,
        "gpu": torch.cuda.get_device_name(0),
        "torch_version": torch.__version__,
        "cuda_version": torch.version.cuda,
        "warmup_runs": warmup_runs,
        "timed_runs": timed_runs,
        "timing_scope": "preprocess + model + camera back-projection",
        "timings_ms": timings_ms,
        "median_time_ms": float(np.median(timings_ms)),
        "valid_fraction": float(valid_mask.mean()),
        "valid_depth_min_m": float(valid_depth.min()),
        "valid_depth_median_m": float(np.median(valid_depth)),
        "valid_depth_max_m": float(valid_depth.max()),
    }
    metadata_path.write_text(json.dumps(metadata, indent=2) + "\n")
    return metadata, visualization_path

print(f"Loaded calibration for: {sorted(intrinsics)}")

## 8. Run the one-frame smoke test

Confirm that the raw depth, gray invalid region, and visualization look
reasonable before enabling the full batch.

In [ ]:
from IPython.display import Image as DisplayImage
from IPython.display import display

metadata, visualization_path = process_frame(
    RECORDING,
    SENSOR,
    IMAGE_INDEX,
    warmup_runs=WARMUP_RUNS,
    timed_runs=TIMED_RUNS,
)
print(json.dumps(metadata, indent=2))
display(DisplayImage(filename=str(visualization_path)))

## 9. Resumable matched batch

Set `RUN_BATCH=True` after the smoke test. The defaults intentionally
match the completed 133-frame UniDAC sequence. Finished frames are
skipped unless `OVERWRITE=True`.

In [ ]:
RUN_BATCH = False
BATCH_RECORDINGS = ["recording2", "recording3", "recording4"]
BATCH_SENSORS = ["G1_A"]
FRAME_STEP = 1
MAX_FRAMES_PER_SENSOR = None
OVERWRITE = False
BATCH_WARMUP_RUNS = 0
BATCH_TIMED_RUNS = 1

if not RUN_BATCH:
    print("Batch is disabled. Set RUN_BATCH=True when ready.")
else:
    completed = 0
    skipped = 0
    for recording in BATCH_RECORDINGS:
        for sensor in BATCH_SENSORS:
            images = find_rgb_images(DATA_DIR, sensor_name=sensor, recording=recording)
            selected_indices = list(range(0, len(images), FRAME_STEP))
            if MAX_FRAMES_PER_SENSOR is not None:
                selected_indices = selected_indices[:MAX_FRAMES_PER_SENSOR]
            for image_index in selected_indices:
                stem = f"{recording}_{sensor}_{image_index:06d}"
                frame_dir = OUTPUT_DIR / recording / sensor
                expected = [
                    frame_dir / f"{stem}_depth_raw.npy",
                    frame_dir / f"{stem}_depth.png",
                    frame_dir / f"{stem}_metadata.json",
                ]
                if not OVERWRITE and all(path.is_file() for path in expected):
                    skipped += 1
                    continue
                metadata, _ = process_frame(
                    recording,
                    sensor,
                    image_index,
                    warmup_runs=BATCH_WARMUP_RUNS,
                    timed_runs=BATCH_TIMED_RUNS,
                )
                completed += 1
                print(
                    f"[{completed}] {recording}/{sensor}/{image_index}: "
                    f"{metadata['median_time_ms']:.1f} ms"
                )

    summary_fields = [
        "recording", "sensor", "image_index", "image_file",
        "timestamp_seconds", "gpu", "median_time_ms", "valid_fraction",
        "valid_depth_min_m", "valid_depth_median_m", "valid_depth_max_m",
        "project_commit", "dac_commit", "variant",
    ]
    summary_rows = []
    for metadata_path in sorted(OUTPUT_DIR.rglob("*_metadata.json")):
        record = json.loads(metadata_path.read_text())
        summary_rows.append({field: record.get(field) for field in summary_fields})
    summary_path = OUTPUT_DIR / "dac_summary.csv"
    with summary_path.open("w", newline="") as summary_file:
        writer = csv.DictWriter(summary_file, fieldnames=summary_fields)
        writer.writeheader()
        writer.writerows(summary_rows)
    print(
        f"Batch complete: {completed} processed, {skipped} skipped. "
        f"Summary: {summary_path}"
    )

## 10. Matched 3D person tracking and sensor evaluation

Run this only after the G1_A DAC batch is complete. It uses the same
YOLO/ByteTrack settings and calibrated ZED/LiDAR references as UniDAC.

In [ ]:
RUN_PERSON_EVALUATION = False
EVALUATION_RECORDINGS = list(BATCH_RECORDINGS)
EVALUATION_FRAME_STEP = FRAME_STEP
EVALUATION_MAX_FRAMES = MAX_FRAMES_PER_SENSOR

if not RUN_PERSON_EVALUATION:
    print(
        "Person evaluation is disabled. Run the G1_A batch, then set "
        "RUN_PERSON_EVALUATION=True."
    )
else:
    DETECTOR_WEIGHTS.parent.mkdir(parents=True, exist_ok=True)
    for recording in EVALUATION_RECORDINGS:
        evaluation_output_dir = OUTPUT_DIR / "evaluation" / recording
        command = [
            sys.executable,
            str(PROJECT_DIR / "evaluate_person_tracking.py"),
            "--data_dir", str(DATA_DIR),
            "--depth_output_dir", str(OUTPUT_DIR),
            "--output_dir", str(evaluation_output_dir),
            "--recording", recording,
            "--frame_step", str(EVALUATION_FRAME_STEP),
            "--detector_weights", str(DETECTOR_WEIGHTS),
            "--device", "0",
        ]
        if EVALUATION_MAX_FRAMES is not None:
            command.extend(["--max_frames", str(EVALUATION_MAX_FRAMES)])
        subprocess.check_call(command, cwd=PROJECT_DIR)
        annotated = sorted((evaluation_output_dir / "annotated").glob("*.jpg"))
        print(f"Evaluation outputs: {evaluation_output_dir}")
        if annotated:
            display(DisplayImage(filename=str(annotated[0])))

## 11. Compare DAC with the completed UniDAC run

Enable this after both evaluation summaries exist. The comparison tool
checks that the recording, frame count, sampling, detector, and physical
reference settings match before producing the table.

In [ ]:
RUN_COMPARISON = False
COMPARISON_RECORDINGS = list(EVALUATION_RECORDINGS)
SUITE_RECORDINGS = ["recording1", *COMPARISON_RECORDINGS]
UNIDAC_OUTPUT_DIR = Path(
    "/content/drive/MyDrive/cv_project_outputs/unidac"
)
COMPARISON_ROOT = Path(
    "/content/drive/MyDrive/cv_project_outputs/comparison"
)

if not RUN_COMPARISON:
    print("Comparison is disabled. Set RUN_COMPARISON=True after both evaluations exist.")
else:
    for recording in COMPARISON_RECORDINGS:
        unidac_summary = (
            UNIDAC_OUTPUT_DIR / "evaluation" / recording / "evaluation_summary.json"
        )
        dac_summary = OUTPUT_DIR / "evaluation" / recording / "evaluation_summary.json"
        for summary in (unidac_summary, dac_summary):
            if not summary.is_file():
                raise FileNotFoundError(f"Missing evaluation summary: {summary}")
        comparison_dir = COMPARISON_ROOT / recording
        comparison_dir.mkdir(parents=True, exist_ok=True)
        comparison_markdown = comparison_dir / "dac_vs_unidac.md"
        comparison_csv = comparison_dir / "dac_vs_unidac.csv"
        subprocess.check_call([
            sys.executable,
            str(PROJECT_DIR / "compare_evaluation_runs.py"),
            "--run", f"UniDAC={unidac_summary}",
            "--run", f"DAC={dac_summary}",
            "--output_csv", str(comparison_csv),
            "--output_markdown", str(comparison_markdown),
        ], cwd=PROJECT_DIR)
        print(comparison_markdown.read_text())
        print(f"Comparison outputs: {comparison_dir}")

    suite_dir = COMPARISON_ROOT / "all_recordings"
    suite_csv = suite_dir / "dac_vs_unidac_suite.csv"
    suite_markdown = suite_dir / "dac_vs_unidac_suite.md"
    subprocess.check_call([
        sys.executable,
        str(PROJECT_DIR / "compare_evaluation_suite.py"),
        "--model_root", f"UniDAC={UNIDAC_OUTPUT_DIR / 'evaluation'}",
        "--model_root", f"DAC={OUTPUT_DIR / 'evaluation'}",
        "--recordings", *SUITE_RECORDINGS,
        "--output_csv", str(suite_csv),
        "--output_markdown", str(suite_markdown),
    ], cwd=PROJECT_DIR)
    print(suite_markdown.read_text())
    print(f"Four-recording summary: {suite_dir}")

## Benchmark note

Use `benchmark_colab.ipynb` for the final speed result. It loads both models
sequentially in one runtime and measures the same decoded G1_A frames on the
same GPU after warm-up. The one-timing-per-frame batch metadata remains useful
for run diagnostics but is not the controlled speed result.